# Inside the Tokenizer

Starter notebook. Fill in the TODOs and **run every cell** before committing — the graders need to see your outputs.


In [ ]:
!pip install tiktoken transformers

In [1]:

# Setup
import tiktoken
from transformers import AutoTokenizer

# Tokenizer 1: tiktoken cl100k_base (BPE, ~100K vocab — used by GPT-4 / GPT-3.5-turbo)
enc_tiktoken = tiktoken.get_encoding("cl100k_base")

# Tokenizer 2: BERT WordPiece (WordPiece, ~30K vocab — fundamentally different algorithm)
# Downloads only tokenizer config + vocab (~few hundred KB, no model weights)
enc_bert = AutoTokenizer.from_pretrained("bert-base-uncased")

print("tiktoken encoding:", enc_tiktoken.name)
print("BERT vocab size  :", enc_bert.vocab_size)
print("Both tokenizers loaded successfully.")

tiktoken encoding: cl100k_base
BERT vocab size  : 30522
Both tokenizers loaded successfully.


## Task 1 — Tokenize and compare

Pick a paragraph of mixed content (English + code + another language + an emoji), tokenize with **two** tokenizers, and show tokens, IDs, and counts side by side.


In [11]:
TEXT = """Machine learning models are powerful. Here is some Python code:
    for i in range(10):
        print(f"step {i}")
Azərbaycan dili çox pratikdir. 🤖✨"""

# --- tiktoken ---
tik_ids    = enc_tiktoken.encode(TEXT)
tik_tokens = [enc_tiktoken.decode([i]) for i in tik_ids]

# --- BERT WordPiece ---
# Note: bert-base-uncased lowercases all input before tokenizing
bert_ids    = enc_bert.encode(TEXT, add_special_tokens=False)
bert_tokens = enc_bert.tokenize(TEXT)

# --- Side-by-side table ---
print(f"{'Tokenizer':<25} {'Token Count':>12}")
print("-" * 38)
print(f"{'tiktoken cl100k_base':<25} {len(tik_ids):>12}")
print(f"{'bert-base-uncased':<25} {len(bert_ids):>12}")

print("\n--- tiktoken tokens (first 40) ---")
print(tik_tokens[:40])

print("\n--- tiktoken IDs (first 40) ---")
print(tik_ids[:40])

print("\n--- BERT tokens (first 40) ---")
print(bert_tokens[:40])

print("\n--- BERT IDs (first 40) ---")
print(bert_ids[:40])


Tokenizer                  Token Count
--------------------------------------
tiktoken cl100k_base                46
bert-base-uncased                   45

--- tiktoken tokens (first 40) ---
['Machine', ' learning', ' models', ' are', ' powerful', '.', ' Here', ' is', ' some', ' Python', ' code', ':\n', '   ', ' for', ' i', ' in', ' range', '(', '10', '):\n', '       ', ' print', '(f', '"', 'step', ' {', 'i', '}")\n', 'Az', 'ə', 'rb', 'ay', 'can', ' d', 'ili', ' ç', 'ox', ' prat', 'ik', 'dir']

--- tiktoken IDs (first 40) ---
[22333, 6975, 4211, 527, 8147, 13, 5810, 374, 1063, 13325, 2082, 512, 262, 369, 602, 304, 2134, 7, 605, 997, 286, 1194, 968, 1, 9710, 314, 72, 14790, 38299, 99638, 10910, 352, 4919, 294, 4008, 18578, 5241, 55744, 1609, 3826]

--- BERT tokens (first 40) ---
['machine', 'learning', 'models', 'are', 'powerful', '.', 'here', 'is', 'some', 'python', 'code', ':', 'for', 'i', 'in', 'range', '(', '10', ')', ':', 'print', '(', 'f', '"', 'step', '{', 'i', '}', '"', ')', 'a

**Where do the two tokenizers disagree most, and why?**

The biggest disagreement appears on the Azerbaijani text and the emoji.
tiktoken uses byte-level BPE with a 100K vocabulary, so it has seen enough
multilingual data to tokenize Azerbaijani words like "Azərbaycan" in 3–4
tokens. BERT's WordPiece vocabulary has only 30K entries and is heavily
English-biased, so it fragments the same word into many more `##`-prefixed
subword pieces — sometimes one token per character. Code and emoji are also
split differently: tiktoken tends to keep common punctuation patterns like
`f"..."` together, while BERT splits aggressively because its vocabulary was
built on natural-language text, not source code.

## Task 2 — Hunt the weird cases

Investigate **at least three**: rare vs common word, English vs another language, code vs prose, leading-space, emoji/accents. Show the token counts that prove your point.


In [12]:
cases = {
    # Case 1: rare/made-up word vs common word of similar length
    "common_word (running)":          "running",
    "rare_word (antidisestablish...)": "antidisestablishmentarianism",

    # Case 2: English sentence vs Azerbaijani translation
    "english_sentence":   "The weather is very nice today.",
    "azerbaijani_sentence": "Bu gün hava çox gözəldir.",

    # Case 3: code snippet vs equivalent prose
    "code_snippet": 'for i in range(10):\n    print(f"step {i}")',
    "prose_equiv":  "Loop from zero to nine and print the step number each iteration.",

    # Case 4: word with and without a leading space
    "no_leading_space ( hello)":   "hello",
    "leading_space   ( hello)":    " hello",

    # Case 5: emoji and accented characters
    "emoji_string":    "🤖✨🎯",
    "accented_string": "naïve café résumé",
}

print(f"{'Case':<40} {'tiktoken':>9} {'BERT':>9}")
print("-" * 60)
for name, s in cases.items():
    n_tik  = len(enc_tiktoken.encode(s))
    n_bert = len(enc_bert.encode(s, add_special_tokens=False))
    print(f"{name:<40} {n_tik:>9} {n_bert:>9}")

Case                                      tiktoken      BERT
------------------------------------------------------------
common_word (running)                            1         1
rare_word (antidisestablish...)                  6         8
english_sentence                                 7         7
azerbaijani_sentence                            12        11
code_snippet                                    15        18
prose_equiv                                     13        13
no_leading_space ( hello)                        1         1
leading_space   ( hello)                         1         1
emoji_string                                     8         1
accented_string                                  7         3


**Observations (one sentence per case):**

**Rare vs common word:** "running" costs 1 token in both tokenizers, while
  "antidisestablishmentarianism" costs 6 in tiktoken and 8 in BERT — BERT
  splits it even more aggressively due to its smaller 30K WordPiece vocabulary,
  meaning jargon-heavy documents will cost noticeably more with BERT-family models.

**English vs Azerbaijani:** The Azerbaijani sentence uses 12 tokens in tiktoken
  vs 7 for the English equivalent (1.7× more), and 11 vs 7 in BERT (1.57×) —
  non-English text consistently consumes more context window and costs more per
  semantic unit of meaning.

**Code vs prose:** The code snippet costs 15 tokens in tiktoken vs 13 for the
  prose equivalent, but the gap widens sharply in BERT (18 vs 13) — BERT was
  trained on natural language and lacks code-aware merge rules, so it fragments
  symbols like `f"..."`, `:`, and indentation into more pieces.

**Leading space:** Both `"hello"` and `" hello"` show a count of 1 in both
  tokenizers — the count looks identical, but the underlying token IDs are
  different, which means the same word at the start vs middle of a sentence is a
  different token and can affect how the model weights that word.

**Emoji and accents:** tiktoken spent 8 tokens on three emoji vs BERT's 1 —
  BERT collapses unknown characters into a single `[UNK]` token which looks
  cheap but destroys meaning entirely; tiktoken byte-encodes each emoji into
  multiple real tokens, preserving the content at higher cost. For accented
  text, BERT used only 3 tokens vs tiktoken's 7 because BERT's vocabulary
  explicitly includes common accented characters like `é` and `ï` as whole tokens.


## Task 3 — Token + cost estimator


In [13]:
def estimate(text, price_in_per_1k, price_out_per_1k, expected_output_tokens):
    """
    Returns (input_token_count, projected_total_cost_usd).
    Uses tiktoken cl100k_base for input token counting.
    """
    n_in  = len(enc_tiktoken.encode(text))
    cost  = (n_in / 1000) * price_in_per_1k + \
            (expected_output_tokens / 1000) * price_out_per_1k
    print(f"  Input tokens     : {n_in}")
    print(f"  Expected output  : {expected_output_tokens} tokens")
    print(f"  Projected cost   : ${cost:.6f}")
    return n_in, cost

In [14]:
# --- Made-up price table (per 1K tokens) ---
PRICE_IN  = 0.075   # $0.075 per 1K input tokens
PRICE_OUT = 0.300   # $0.300 per 1K output tokens

In [15]:
# Input 1: short question
short_q = "How do I reset my password?"

In [16]:
# Input 2: long document (simulate a multi-paragraph text)
long_doc = """\
Artificial intelligence (AI) is intelligence demonstrated by machines,
as opposed to the natural intelligence displayed by animals including humans.
AI research has been defined as the field of study of intelligent agents,
which refers to any system that perceives its environment and takes actions
that maximize its chance of achieving its goals.

The term "artificial intelligence" had previously been used to describe
machines that mimic and display "human" cognitive skills associated with the
human mind, such as "learning" and "problem-solving". This definition has
since been rejected by major AI researchers who now describe AI in terms of
rationality and acting rationally, which does not limit how intelligence can
be articulated.

AI applications include advanced web search engines, recommendation systems,
understanding human speech, self-driving cars, generative or creative tools,
and competing at the highest level in strategic games. As machines become
increasingly capable, tasks considered to require intelligence are often
removed from the definition of AI, a phenomenon known as the AI effect.
""" * 3   # repeat 3x to make it truly long


In [17]:
# Input 3: code file (a realistic Python script snippet)
code_file = """\
import os
import json
from pathlib import Path

def load_config(path: str) -> dict:
    \"\"\"Load a JSON config file and return it as a dict.\"\"\"
    config_path = Path(path)
    if not config_path.exists():
        raise FileNotFoundError(f"Config not found: {path}")
    with open(config_path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_results(results: list, output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)
    out_file = Path(output_dir) / "results.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"Saved {len(results)} results to {out_file}")

if __name__ == "__main__":
    cfg = load_config("config.json")
    results = [{"id": i, "value": i ** 2} for i in range(100)]
    save_results(results, cfg.get("output_dir", "outputs"))
"""


In [18]:
inputs = {
    "Short question" : (short_q,    50),
    "Long document"  : (long_doc,  300),
    "Code file"      : (code_file, 150),
}

In [19]:
print(f"{'='*50}")
print(f"Price table:  in=${PRICE_IN}/1K  out=${PRICE_OUT}/1K")
print(f"{'='*50}\n")
results_table = []
for label, (text, expected_out) in inputs.items():
    print(f"[{label}]")
    n, cost = estimate(text, PRICE_IN, PRICE_OUT, expected_out)
    results_table.append((label, n, cost))
    print()

print(f"\n{'Input':<20} {'Tokens':>8} {'Cost ($)':>12}")
print("-" * 42)
for label, n, cost in results_table:
    print(f"{label:<20} {n:>8} {cost:>12.6f}")

Price table:  in=$0.075/1K  out=$0.3/1K

[Short question]
  Input tokens     : 7
  Expected output  : 50 tokens
  Projected cost   : $0.015525

[Long document]
  Input tokens     : 627
  Expected output  : 300 tokens
  Projected cost   : $0.137025

[Code file]
  Input tokens     : 221
  Expected output  : 150 tokens
  Projected cost   : $0.061575


Input                  Tokens     Cost ($)
------------------------------------------
Short question              7     0.015525
Long document             627     0.137025
Code file                 221     0.061575


**Which input is most expensive, and is it what you expected?**

The long document is the most expensive at $0.137 — nearly 9× the cost of the
short question ($0.015) — which is expected given its 627 input tokens vs just 7.
What is notable is that the code file, despite being a realistic multi-function
Python script, costs only $0.061: at 221 tokens it "compresses" well because BPE
has learned common keywords like `import`, `def`, `os.makedirs`, and `json.load`
as single tokens. The short question is almost free individually, which confirms
that conversational API calls carry negligible cost — the real budget risk is bulk
document processing where hundreds of long-doc calls stack up fast.